## scraping content from URLs

In [ ]:
import pandas as pd
import urllib3
from urllib3 import Retry
from tqdm.notebook import tqdm
import time
from bs4 import BeautifulSoup

serp_df = pd.read_csv("Data/logs_compiled.csv")
len(serp_df["url"].unique())

corpus = []
for idx, url in tqdm(zip(range(len(serp_df["url"].unique())), serp_df["url"].unique()), total = len(serp_df["url"].unique())):
    if idx % 50 == 0 and idx!=0:
        time.sleep(10)
    if url!=None:
        try:
            html = urllib3.request("GET", url, retries=Retry(5))
            # html = urllib3.request("GET", url, retries=False)
            if html.status == 200:
                soup = BeautifulSoup(markup = html.data, features='lxml')
                # kill all script and style elements
                for script in soup(["script", "style", "nav", "header", "footer", "aside", "img", "link"]):
                    script.extract() # rip it out
                    # script.decompose() 
                corpus.append(["d"+str(idx), url, str(soup.body), str(soup.body.get_text(separator='\n ', strip=True))])
            else:
                corpus.append(["d"+str(idx), url, None, None])
        except:
            corpus.append(["d"+str(idx), url, None, None])

  0%|          | 0/222 [00:00<?, ?it/s]

In [3]:
corpus_df = pd.DataFrame(corpus, columns = ["doc_id", "url", "html_content", "text_content"])
corpus_df.to_excel("Data/corpus.xlsx", index=False)

In [ ]:
corpus_df = pd.read_excel("Data/corpus.xlsx")

text_content = []

for idx, row in corpus_df.iterrows():
    if pd.isna(row["text_content"]):
        try:
            html_content = row["html_content"]
            soup = BeautifulSoup(markup = html_content, features='lxml')
            # kill all script and style elements
            for script in soup(["script", "style", "nav", "header", "footer", "aside", "img", "link"]):
                script.extract() # rip it out
                # script.decompose() 
            row.iloc[idx]["text_content"] = str(soup.body.get_text(separator='\n ', strip=True))
        except:
            continue

In [5]:
corpus_df.head()

,doc_id,url,html_content,text_content
0,d0,https://wwftravel.it/blog/viaggi/india-natura-...,"<body class=""wp-singular post-template-default...",Vai al contenuto\n Home\n Blog\n Viaggi\n Indi...
1,d1,https://www.terreincognitemagazine.it/natura-s...,"<body class=""single single-post postid-2526 si...",Homepage\n >\n CULTURA\n >\n MOSTRE\n >\n Natu...
2,d2,https://www.storicang.it/a/civilta-dellindo-i-...,None,None
3,d3,https://www.focusjunior.it/scuola/storia/chi-s...,"<body><div id=""fb-root""></div>\n<noscript></no...",FocusJunior.it\n Scuola\n Storia\n Chi sono gl...
4,d4,https://www.focusjunior.it/comportamento/feste...,"<body><div id=""fb-root""></div>\n<noscript></no...",FocusJunior.it\n Comportamento\n Feste\n Festa...


In [6]:
len(corpus_df.loc[pd.isna(corpus_df["text_content"])]), len(corpus_df.loc[pd.isna(corpus_df["html_content"])])

(11, 11)